In [1]:

# STEP 4.1: Load and validate cleaned datasets

import pandas as pd
import numpy as np
from pathlib import Path

# Cleaned data is one folder above this notebook
data_dir = Path("../Clean Data")

# Load the cleaned datasets
transactions_fe = pd.read_csv(
    data_dir / "finlora_transactions_clean.csv",
    parse_dates=["timestamp", "account_created_date"]
)

accounts_fe = pd.read_csv(
    data_dir / "finlora_accounts_clean.csv",
    parse_dates=["account_created_date"]
)

# Create a working copy for feature engineering
model_data = transactions_fe.copy()

# Validate datasets
assert model_data.shape == (126000, 29)
assert accounts_fe.shape == (7200, 8)
assert model_data["transaction_id"].is_unique
assert model_data["is_fraud"].sum() == 3352

print("FEATURE ENGINEERING INITIALIZATION")
print("-" * 50)

print("Transactions:", transactions_fe.shape)
print("Accounts:", accounts_fe.shape)
print("Working dataset:", model_data.shape)
print("Fraud cases:", model_data["is_fraud"].sum())

print("\nTimestamp data type:", model_data["timestamp"].dtype)

print("\nFEATURE ENGINEERING DATA READY")


FEATURE ENGINEERING INITIALIZATION
--------------------------------------------------
Transactions: (126000, 29)
Accounts: (7200, 8)
Working dataset: (126000, 29)
Fraud cases: 3352

Timestamp data type: datetime64[ns, UTC]

FEATURE ENGINEERING DATA READY


In [2]:
model_data.head()

,transaction_id,account_id,account_type,kyc_tier,timestamp,day_of_week,hour_of_day,description,merchant_name,merchant_category,...,device_id,is_new_device,account_age_days,status,is_fraud,device_info_missing,is_negative_amount,is_negative_baseline,account_created_date,account_date_inconsistent
0,FLR250216186265,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-02-16 14:55:35+00:00,Sunday,14,CNP PURCHASE - BOLT,Bolt,Travel,...,NaN,NaN,903,Completed,0,1,0,0,2022-08-28 00:00:00+00:00,0
1,FLR250423143305,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-23 13:35:40+00:00,Wednesday,13,WEB PURCHASE - SLACK,Slack,Subscription/SaaS,...,NaN,NaN,969,Declined,0,1,0,0,2022-08-28 00:00:00+00:00,0
2,FLR250424154897,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-24 12:30:34+00:00,Thursday,12,MOBILE PURCHASE - JUSTRITE SUPERSTORE,Justrite Superstore,Groceries,...,DEV-9055235108,0.0,970,Completed,0,0,0,0,2022-08-28 00:00:00+00:00,0
3,FLR250428101772,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-28 14:41:59+00:00,Monday,14,CNP PURCHASE - ADOBE CREATIVE CLOUD,Adobe Creative Cloud,Subscription/SaaS,...,NaN,NaN,974,Completed,0,1,0,0,2022-08-28 00:00:00+00:00,0
4,FLR250628102827,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-06-28 16:15:47+00:00,Saturday,16,MOBILE PURCHASE - APPLE STORE,Apple Store,Electronics,...,DEV-9055235108,0.0,1035,Declined,0,0,0,0,2022-08-28 00:00:00+00:00,0


In [3]:
model_data.columns

Index(['transaction_id', 'account_id', 'account_type', 'kyc_tier', 'timestamp',
       'day_of_week', 'hour_of_day', 'description', 'merchant_name',
       'merchant_category', 'channel', 'amount', 'currency',
       'amount_to_avg_ratio', 'avg_transaction_amount_30d',
       'transaction_velocity_1h', 'transaction_country', 'home_country',
       'is_cross_border', 'device_id', 'is_new_device', 'account_age_days',
       'status', 'is_fraud', 'device_info_missing', 'is_negative_amount',
       'is_negative_baseline', 'account_created_date',
       'account_date_inconsistent'],
      dtype='object')

In [5]:

# STEP 4.2: Inspect columns for feature selection

print("FINLORA — FEATURE SELECTION INSPECTION")
print("-" * 50)

print("\nALL AVAILABLE COLUMNS")

for index, column in enumerate(model_data.columns, start=1):
    print(f"{index:2}. {column}")

# Exclude  columns not relevant for modelling or that can lead to data leakage.
excluded_columns = [
    "transaction_id",
    "account_id",
    "device_id",
    "status",
    "account_age_days",
    "account_date_inconsistent",
    "account_created_date",
    "is_fraud"
]

# Check that the exclusion list matches the dataset
missing_exclusions = [
    col for col in excluded_columns
    if col not in model_data.columns
]

candidate_columns = [
    col for col in model_data.columns
    if col not in excluded_columns
]

print("\nEXCLUDED COLUMNS")
print(excluded_columns)




FINLORA — FEATURE SELECTION INSPECTION
--------------------------------------------------

ALL AVAILABLE COLUMNS
 1. transaction_id
 2. account_id
 3. account_type
 4. kyc_tier
 5. timestamp
 6. day_of_week
 7. hour_of_day
 8. description
 9. merchant_name
10. merchant_category
11. channel
12. amount
13. currency
14. amount_to_avg_ratio
15. avg_transaction_amount_30d
16. transaction_velocity_1h
17. transaction_country
18. home_country
19. is_cross_border
20. device_id
21. is_new_device
22. account_age_days
23. status
24. is_fraud
25. device_info_missing
26. is_negative_amount
27. is_negative_baseline
28. account_created_date
29. account_date_inconsistent

EXCLUDED COLUMNS
['transaction_id', 'account_id', 'device_id', 'status', 'account_age_days', 'account_date_inconsistent', 'account_created_date', 'is_fraud']


In [10]:

# STEP 4.3: Create the initial modeling feature set

# Numerical predictors
numerical_features = [
    "amount",
    "amount_to_avg_ratio",
    "transaction_velocity_1h",
    "hour_of_day",
    "is_cross_border",
    "is_new_device",
    "device_info_missing",
    "is_negative_amount",
    "is_negative_baseline"
]

# Categorical predictors
categorical_features = [
    "currency",
    "account_type",
    "kyc_tier",
    "merchant_category",
    "channel"
]

# Combine selected features
selected_features = numerical_features + categorical_features

# Create initial feature matrix
X = model_data[selected_features].copy()

# Prediction target
y = model_data["is_fraud"].copy()

# Preserve metadata separately for splitting and auditing
model_metadata = model_data[
    ["transaction_id", "account_id", "timestamp"]
].copy()





In [13]:

# STEP 4.4: Create transformed numerical features

import numpy as np

# Work on a copy of the selected feature matrix
X_engineered = X.copy()

# 1. Signed log transformation for transaction amount
X_engineered["signed_log_amount"] = (
    np.sign(X_engineered["amount"])
    * np.log1p(np.abs(X_engineered["amount"]))
)

# 2. Log transformation for spending ratio
X_engineered["log_spending_ratio"] = np.log1p(
    X_engineered["amount_to_avg_ratio"]
)

# 3. Cyclical encoding for hour of day
X_engineered["hour_sin"] = np.sin(
    2 * np.pi * X_engineered["hour_of_day"] / 24
)

X_engineered["hour_cos"] = np.cos(
    2 * np.pi * X_engineered["hour_of_day"] / 24
)

print("FEATURE TRANSFORMATION")
print("-" * 50)

print("Original feature count:", X.shape[1])
print("Engineered feature count:", X_engineered.shape[1])

print("\nNEW FEATURES")
new_features = [
    "signed_log_amount",
    "log_spending_ratio",
    "hour_sin",
    "hour_cos"
]

display(X_engineered[new_features].isna().sum())


print("Engineered X shape:", X_engineered.shape)


FEATURE TRANSFORMATION
--------------------------------------------------
Original feature count: 14
Engineered feature count: 18

NEW FEATURES


signed_log_amount     0
log_spending_ratio    0
hour_sin              0
hour_cos              0
dtype: int64

Engineered X shape: (126000, 18)


In [14]:

# STEP 4.5: Stratified train-test split

from sklearn.model_selection import train_test_split

# Split row indices to keep features, labels,
# and transaction metadata perfectly aligned.
train_idx, test_idx = train_test_split(
    X_engineered.index,
    test_size=0.25,
    stratify=y,
    random_state=42
)

# Create training and testing feature sets
X_train = X_engineered.loc[train_idx].copy()
X_test = X_engineered.loc[test_idx].copy()

# Create training and testing targets
y_train = y.loc[train_idx].copy()
y_test = y.loc[test_idx].copy()

# Preserve transaction metadata
metadata_train = model_metadata.loc[train_idx].copy()
metadata_test = model_metadata.loc[test_idx].copy()

print(" STRATIFIED TRAIN-TEST SPLIT")
print("-" * 50)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTRAINING TARGET DISTRIBUTION")
display(y_train.value_counts().sort_index())

print("\nTESTING TARGET DISTRIBUTION")
display(y_test.value_counts().sort_index())

print("\nFRAUD RATES")

print(f"Overall fraud rate: {y.mean() * 100:.2f}%")
print(f"Training fraud rate: {y_train.mean() * 100:.2f}%")
print(f"Testing fraud rate: {y_test.mean() * 100:.2f}%")

print("\nDATA INTEGRITY CHECKS")

assert len(X_train) == len(y_train) == len(metadata_train)
assert len(X_test) == len(y_test) == len(metadata_test)

assert len(X_train) + len(X_test) == len(X_engineered)

assert set(train_idx).isdisjoint(set(test_idx))

print("All split integrity checks passed.")

 STRATIFIED TRAIN-TEST SPLIT
--------------------------------------------------
Training features: (94500, 18)
Testing features: (31500, 18)

TRAINING TARGET DISTRIBUTION


is_fraud
0    91986
1     2514
Name: count, dtype: int64


TESTING TARGET DISTRIBUTION


is_fraud
0    30662
1      838
Name: count, dtype: int64


FRAUD RATES
Overall fraud rate: 2.66%
Training fraud rate: 2.66%
Testing fraud rate: 2.66%

DATA INTEGRITY CHECKS
All split integrity checks passed.


In [15]:

# STEP 4.6:  Leakage-safe preprocessing pipelines



from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)
from sklearn.impute import SimpleImputer

# Continuous features to scale
continuous_features = [
    "signed_log_amount",
    "log_spending_ratio",
    "transaction_velocity_1h",
    "hour_sin",
    "hour_cos"
]

# Binary indicators to preserve
binary_features = [
    "is_cross_border",
    "device_info_missing",
    "is_negative_amount",
    "is_negative_baseline"
]

# Categorical features
categorical_model_features = [
    "currency",
    "account_type",
    "kyc_tier",
    "merchant_category",
    "channel",
    "is_new_device"
]

# Numerical preprocessing
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Device novelty: distinguish missing from 0 and 1
# Convert to strings so OneHotEncoder can handle categories.
def prepare_device_category(data):
    data = data.copy()
    data["is_new_device"] = (
        data["is_new_device"]
        .map({0.0: "Known", 1.0: "New"})
        .fillna("Unknown")
    )
    return data

# Create model-ready views without modifying originals
X_train_model = prepare_device_category(X_train)
X_test_model = prepare_device_category(X_test)

# Categorical preprocessing
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Unknown"
    )),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine transformations
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, continuous_features),
        ("binary", "passthrough", binary_features),
        (
            "categorical",
            categorical_pipeline,
            categorical_model_features
        )
    ],
    remainder="drop"
)

print(" PREPROCESSING CONFIGURATION")
print("-" * 50)

print("Continuous features:", len(continuous_features))
print("Binary features:", len(binary_features))
print("Categorical features:", len(categorical_model_features))

print("\nTraining dataset:", X_train_model.shape)
print("Testing dataset:", X_test_model.shape)

print("\nDEVICE NOVELTY DISTRIBUTION — TRAINING")
display(
    X_train_model["is_new_device"].value_counts()
)

print("\nMISSING DEVICE CATEGORIES")
print(
    X_train_model["is_new_device"].isna().sum()
)

print("\nPreprocessor defined successfully.")
print("No preprocessing has been fitted yet.")


 PREPROCESSING CONFIGURATION
--------------------------------------------------
Continuous features: 5
Binary features: 4
Categorical features: 6

Training dataset: (94500, 18)
Testing dataset: (31500, 18)

DEVICE NOVELTY DISTRIBUTION — TRAINING


is_new_device
Known      80791
Unknown    10751
New         2958
Name: count, dtype: int64


MISSING DEVICE CATEGORIES
0

Preprocessor defined successfully.
No preprocessing has been fitted yet.


In [17]:

# STEP 4.7: Fit and validate preprocessing

from scipy import sparse


# Fit ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train_model)

# Transform test data using the fitted preprocessor
X_test_processed = preprocessor.transform(X_test_model)

# Obtain final encoded feature names
processed_feature_names = preprocessor.get_feature_names_out()

print("PREPROCESSING VALIDATION")
print("-" * 50)

print("Training processed shape:", X_train_processed.shape)
print("Testing processed shape:", X_test_processed.shape)
print("Total processed features:", len(processed_feature_names))

print("\nVALIDATION CHECKS")

assert X_train_processed.shape[0] == len(y_train)
assert X_test_processed.shape[0] == len(y_test)

assert X_train_processed.shape[1] == X_test_processed.shape[1]
assert X_train_processed.shape[1] == len(processed_feature_names)

# Check for invalid values in dense or sparse outputs
train_values = (
    X_train_processed.data
    if sparse.issparse(X_train_processed)
    else X_train_processed
)

test_values = (
    X_test_processed.data
    if sparse.issparse(X_test_processed)
    else X_test_processed
)






PREPROCESSING VALIDATION
--------------------------------------------------
Training processed shape: (94500, 42)
Testing processed shape: (31500, 42)
Total processed features: 42

VALIDATION CHECKS


In [19]:

# STEP 4.8: Save model-ready datasets and preprocessing

from pathlib import Path
from scipy import sparse
import joblib


# Create a dedicated folder for modeling artifacts
artifact_dir = Path("../Model Development/artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

# Save processed feature matrices
sparse.save_npz(
    artifact_dir / "X_train_processed.npz",
    sparse.csr_matrix(X_train_processed)
)

sparse.save_npz(
    artifact_dir / "X_test_processed.npz",
    sparse.csr_matrix(X_test_processed)
)

# Save target labels
np.save(
    artifact_dir / "y_train.npy",
    y_train.to_numpy()
)

np.save(
    artifact_dir / "y_test.npy",
    y_test.to_numpy()
)

# Save the fitted preprocessing pipeline
joblib.dump(
    preprocessor,
    artifact_dir / "preprocessor.joblib"
)

# Save processed feature names
np.save(
    artifact_dir / "processed_feature_names.npy",
    processed_feature_names
)

# Save transaction metadata for later validation
metadata_train.to_csv(
    artifact_dir / "metadata_train.csv",
    index=False
)

metadata_test.to_csv(
    artifact_dir / "metadata_test.csv",
    index=False
)

print("MODELING ARTIFACT EXPORT")
print("-" * 50)

print("Artifacts saved to:", artifact_dir.resolve())

print("\nSAVED FILES")

for file in sorted(artifact_dir.iterdir()):
    print("-", file.name)

print("\nTraining matrix:", X_train_processed.shape)
print("Testing matrix:", X_test_processed.shape)

print("\nARTIFACT EXPORT COMPLETED")


MODELING ARTIFACT EXPORT
--------------------------------------------------
Artifacts saved to: C:\Users\osato\OneDrive\Dokumente\Documents\Amdari\Project 1\Model Development\artifacts

SAVED FILES
- metadata_test.csv
- metadata_train.csv
- preprocessor.joblib
- processed_feature_names.npy
- X_test_processed.npz
- X_train_processed.npz
- y_test.npy
- y_train.npy

Training matrix: (94500, 42)
Testing matrix: (31500, 42)

ARTIFACT EXPORT COMPLETED


In [20]:

# Fix the saved feature names data type

import numpy as np
from pathlib import Path

artifact_dir = Path("../Model Development/artifacts")

# Convert feature names to standard strings
feature_names_safe = np.asarray(
    processed_feature_names,
    dtype=str
)

# Overwrite only the feature names file
np.save(
    artifact_dir / "processed_feature_names.npy",
    feature_names_safe
)

print("Feature names saved successfully.")
print("Data type:", feature_names_safe.dtype)
print("Number of features:", len(feature_names_safe))


Feature names saved successfully.
Data type: <U48
Number of features: 42
